# 05 · Avaliação

**CRISP-DM fase 5, avaliação** · Turismo e Hotelaria

---

Esta fase avalia o projeto como um todo e o modelo selecionado sob a ótica do
negócio: a diretoria do resort e do hotel urbano consegue usar os segmentos
encontrados para decidir overbooking, canal de venda e antecedência?

!!! success "Resumo executivo"
    **Aprovado para implantação.** O K-Means com 5 segmentos (escolhido no `04-0`)
    atende 5 dos 6 critérios do [Canvas do Problema](../docs/pbl/canvas-do-problema.md):
    a taxa de cancelamento varia 6,6x entre o segmento mais arriscado e o mais seguro
    (estatisticamente significativo), e uma reserva nova é atribuída ao segmento
    certo em 99,7% dos casos. O único critério que falha é a silhueta (0,124, abaixo
    do corte de 0,25), não crítico, e já esperado desde o `04-1`. No cenário
    financeiro base (hipotético), o investimento se paga no primeiro ano.

| Etapa | Entrega |
|---|---|
| **5.1 Avaliar os resultados** | modelo validado contra os critérios do Canvas do Problema · impacto financeiro e operacional estimado · decisão formal sobre a implantação |
| **5.2 Revisar o processo** | análise das fases concluídas · lacunas e erros documentados · sumário das descobertas e limitações |
| **5.3 Determinar as próximas etapas** | próxima ação (Implantação, Iterar mais ou Novo projeto) · justificativa formalizada |

Entradas:
* os segmentos e o resumo do modelo vencedor, salvos pelo [`04-0-modelagem-cluster`](04-0-modelagem-cluster.ipynb) (K-Means, k=5);
* a base pré-processada do [`03-preparacao`](03-preparacao.ipynb);
* a base tratada, que dá as unidades de negócio, e o rótulo reservado `is_canceled`.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import silhouette_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

## Critérios e premissas

**Confira estes valores com os itens 4 e 6 do [Canvas do Problema](../docs/pbl/canvas-do-problema.md).** Eles definem se o modelo é aprovado.

* **Críticos:** traduzem o uso do modelo pela operação. Se algum falhar, o modelo não vai para implantação.
* **Não críticos:** qualidade técnica. Se falharem, o modelo pode seguir, mas com ressalva.

As premissas financeiras são **hipotéticas** e precisam ser validadas com a operação antes de qualquer decisão.

In [ ]:
# Critérios de sucesso do negócio (Canvas, itens 4 e 6)
K_MIN, K_MAX = 2, 5  # crítico: nº de segmentos que a operação consegue atender com ação própria
# crítico: menor segmento com ao menos 10% da carteira, senão não paga uma ação dedicada
FATIA_MINIMA = 0.10
# crítico: taxa de cancelamento do segmento mais arriscado >= 1,3x a do mais seguro
RAZAO_CANCELAMENTO_MINIMA = 1.30
ACURACIA_MINIMA = 0.85  # crítico: reserva nova cai no segmento certo em ao menos 85% dos casos
SILHUETA_MINIMA = 0.25  # não crítico: separação geométrica dos grupos (mesmo corte usado no 04-1)

# Premissas financeiras (hipotéticas: validar com a operação do resort e do hotel urbano)
# ganho de receita atribuído à segmentação
GANHO_RECEITA = {"pessimista": 0.01, "base": 0.03, "otimista": 0.05}
CUSTO_IMPLANTACAO = 80_000  # US$, uma vez: integração com o PMS, treinamento da equipe de reservas
CUSTO_ANUAL = 25_000  # US$ por ano: manutenção e re-treino do modelo

In [ ]:
# saída do 03: já sem is_canceled
base = pd.read_csv(RAIZ / "data" / "processed" / "hotel_bookings_preprocessed.csv", sep=";")
X = base.to_numpy(dtype=float)

# base tratada, em unidades de negócio
df = pd.read_csv(RAIZ / "data" / "processed" / "hotel_bookings_tratado.csv", sep=";")
# rótulos do 04-0
caminho_segmentos = RAIZ / "data" / "processed" / "segmentos.csv"
segmentos = pd.read_csv(caminho_segmentos, sep=";")["segmento"].to_numpy()
# nome, parâmetros e silhueta do vencedor
selecionado = json.loads((RAIZ / "models" / "selecionado.json").read_text())

assert len(X) == len(df) == len(segmentos), "base, tratado e segmentos precisam estar alinhados"
# -1 seria ruído (só em algoritmo de densidade); o K-Means não produz ruído
validos = segmentos >= 0

display(
    Markdown(
        f"Modelo selecionado: **{selecionado['modelo']}** · `{selecionado['params']}` · "
        f"**{selecionado['grupos']}** segmentos · silhueta **{selecionado['silhueta']:.3f}**"
    )
)

# 5.1 · Avaliar os resultados (negócio)

## Os segmentos em unidades de negócio

Os segmentos são renomeados de A em diante, **do maior risco de cancelamento para o
menor**. A numeração do algoritmo (0 a 4) é arbitrária e não deve chegar à equipe de
reservas: o que importa para a operação é o risco, não o número interno do cluster.

In [ ]:
ordem = (
    pd.Series(df.loc[validos, "is_canceled"].to_numpy(), index=segmentos[validos])
    .groupby(level=0)
    .mean()
    .sort_values(ascending=False)
    .index
)  # segmentos do maior cancelamento ao menor
nome = {s: f"Segmento {chr(65 + i)}" for i, s in enumerate(ordem)}  # 0..4 -> Segmento A..E
# ruído (se houvesse) viraria "Sem segmento"
df["segmento"] = pd.Series(segmentos).map(nome).fillna("Sem segmento")

perfil = (
    df.groupby("segmento")
    .agg(
        reservas=("is_canceled", "size"),
        taxa_cancelamento=("is_canceled", "mean"),
        lead_time_mediano=("lead_time", "median"),
        noites_mediana=("total_nights", "median"),
        adr_mediano=("adr", "median"),
        solicitacoes_mediana=("total_of_special_requests", "median"),
    )
    .assign(participacao=lambda tabela: tabela["reservas"] / len(df))
    .sort_values("taxa_cancelamento", ascending=False)
)
perfil.style.format(
    {
        "taxa_cancelamento": "{:.1%}",
        "participacao": "{:.1%}",
        "lead_time_mediano": "{:.0f}",
        "noites_mediana": "{:.0f}",
        "adr_mediano": "{:.2f}",
        "solicitacoes_mediana": "{:.0f}",
    }
)

In [ ]:
fig, (ax_canc, ax_fatia) = plt.subplots(
    1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]}
)

ordem_nomes = [n for n in perfil.index if n != "Sem segmento"]
# risco de cancelamento por segmento
perfil.loc[ordem_nomes, "taxa_cancelamento"].plot.bar(ax=ax_canc, color="tab:red")
ax_canc.set(title="Taxa de cancelamento por segmento", ylabel="taxa de cancelamento")
ax_canc.tick_params(axis="x", rotation=0)
ax_canc.grid(alpha=0.3, axis="y")

perfil["participacao"].plot.barh(ax=ax_fatia, color="tab:blue")  # tamanho de cada segmento
ax_fatia.axvline(FATIA_MINIMA, color="tab:red", ls="--", label=f"mínimo ({FATIA_MINIMA:.0%})")
ax_fatia.set(title="Participação na carteira", xlabel="")
ax_fatia.invert_yaxis()
ax_fatia.legend()
fig.tight_layout()
plt.show()

## Validação contra os critérios de sucesso (Canvas, itens 4 e 6)

* **Cancelamento:** o teste qui-quadrado verifica se segmento e cancelamento são
  independentes; a razão entre a maior e a menor taxa mede se a diferença é grande
  o bastante para justificar uma política de overbooking diferente por segmento.
* **Reserva nova classificável:** uma random forest aprende a atribuir o segmento a
  partir das mesmas features. A acurácia em validação cruzada estima quantos pedidos
  de reserva novos seriam classificados corretamente, o que viabiliza usar o modelo
  no dia a dia do motor de reservas.

In [ ]:
rotulos = segmentos[validos]
k = len(np.unique(rotulos))  # segmentos formados (sem ruído)
fatia_min = np.bincount(rotulos).min() / len(df)  # tamanho do menor segmento
silhueta = silhouette_score(X[validos], rotulos)  # recalculada na base atual

tabela_contingencia = pd.crosstab(segmentos[validos], df.loc[validos, "is_canceled"])
# p < 0,05: segmento e cancelamento não são independentes (estatística qui-quadrado descartada)
_, p_valor, _, _ = chi2_contingency(tabela_contingencia)

taxas = df.loc[validos].groupby(segmentos[validos])["is_canceled"].mean()
razao_cancelamento = taxas.max() / taxas.min()  # segmento mais arriscado / segmento mais seguro

classificador = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)
acuracia = cross_val_score(
    classificador, X[validos], rotulos, cv=StratifiedKFold(5, shuffle=True, random_state=42)
).mean()  # acerto em reservas fora do treino

cancelamento_ok = razao_cancelamento >= RAZAO_CANCELAMENTO_MINIMA and p_valor < 0.05
cobertura = validos.mean()

avaliacao = pd.DataFrame(
    [
        ("Segmentos operáveis", f"{K_MIN} a {K_MAX}", f"{k}", K_MIN <= k <= K_MAX, True),
        (
            "Menor segmento",
            f">= {FATIA_MINIMA:.0%} da carteira",
            f"{fatia_min:.1%}",
            fatia_min >= FATIA_MINIMA,
            True,
        ),
        (
            "Cancelamento distingue os segmentos",
            f"razão >= {RAZAO_CANCELAMENTO_MINIMA} e p < 0,05",
            f"{razao_cancelamento:.2f} (p = {p_valor:.1e})",
            cancelamento_ok,
            True,
        ),
        (
            "Reserva nova classificável",
            f">= {ACURACIA_MINIMA:.0%}",
            f"{acuracia:.1%}",
            acuracia >= ACURACIA_MINIMA,
            True,
        ),
        (
            "Separação dos grupos",
            f"silhueta >= {SILHUETA_MINIMA}",
            f"{silhueta:.3f}",
            silhueta >= SILHUETA_MINIMA,
            False,
        ),
        ("Cobertura da carteira", "100% com segmento", f"{cobertura:.1%}", cobertura == 1, False),
    ],
    columns=["critério", "meta", "obtido", "atende", "crítico"],
)


def _destacar_falhas(linha):
    cor = "background-color: #fde2e1" if linha["atende"] == "falhou" else ""
    return [cor] * len(linha)


(
    avaliacao.assign(
        atende=avaliacao["atende"].map({True: "ok", False: "falhou"}),
        critico=avaliacao["crítico"].map({True: "sim", False: "não"}),
    )
    .drop(columns="crítico")
    .style.hide(axis="index")
    .apply(_destacar_falhas, axis=1)
)

## Impacto financeiro e operacional

**Financeiro.** A base de cálculo é a receita de hospedagem realizada: a soma de
`adr x total_nights` das reservas não canceladas, anualizada pelos anos que a base
cobre. Cada cenário aplica um ganho de receita atribuído à segmentação, vindo de uma
política de overbooking calibrada por segmento (menos quartos vagos por cancelamento
mal previsto, menos *walk* por overbooking excessivo). O resultado do ano 1 desconta
a implantação e a manutenção.

In [ ]:
anos = df["arrival_date_year"].nunique()  # a base cobre chegadas de meados de 2015 a meados de 2017
receita_anual = df.loc[validos & (df["is_canceled"] == 0)].eval("adr * total_nights").sum() / anos

cenarios = pd.DataFrame({"ganho": GANHO_RECEITA}).assign(
    receita_extra=lambda tabela: receita_anual * tabela["ganho"],
    resultado_ano1=lambda tabela: tabela["receita_extra"] - CUSTO_IMPLANTACAO - CUSTO_ANUAL,
    roi_ano1=lambda tabela: tabela["resultado_ano1"] / (CUSTO_IMPLANTACAO + CUSTO_ANUAL),
    payback_meses=lambda tabela: (
        CUSTO_IMPLANTACAO / ((tabela["receita_extra"] - CUSTO_ANUAL) / 12)
    ).where(tabela["receita_extra"] > CUSTO_ANUAL),
)
texto_receita = f"Receita anual de hospedagem (reservas não canceladas): **${receita_anual:,.0f}**"
display(Markdown(texto_receita))
cenarios.style.format(
    {
        "ganho": "{:.0%}",
        "receita_extra": "${:,.0f}",
        "resultado_ano1": "${:,.0f}",
        "roi_ano1": "{:.0%}",
        "payback_meses": "{:.1f}",
    },
    na_rep="sem payback",
)

In [ ]:
operacional = pd.DataFrame(
    {
        "indicador": [
            "Políticas de overbooking a manter",
            "Reservas da carteira com segmento",
            "Reservas novas classificadas corretamente",
            "Maior segmento (concentração da carga)",
            "Re-treino recomendado",
        ],
        "valor": [
            f"{k} (uma por segmento)",
            f"{validos.mean():.1%}",
            f"{acuracia:.1%}",
            f"{perfil['participacao'].max():.1%} da carteira",
            "semestral ou a cada 10% de reservas novas",
        ],
    }
)
operacional.style.hide(axis="index")

## Decisão formal sobre a implantação

In [ ]:
falhas_criticas = avaliacao.query("crítico and not atende")["critério"].tolist()
ressalvas = avaliacao.query("not crítico and not atende")["critério"].tolist()
cenario_base_positivo = cenarios.loc["base", "resultado_ano1"] > 0

aprovado = not falhas_criticas and cenario_base_positivo
display(
    Markdown(
        f"### {'Aprovado' if aprovado else 'Não aprovado'} para implantação\n"
        f"- critérios críticos que falharam: **{', '.join(falhas_criticas) or 'nenhum'}**\n"
        f"- ressalvas (não críticos): **{', '.join(ressalvas) or 'nenhuma'}**\n"
        f"- resultado do ano 1 no cenário base: **${cenarios.loc['base', 'resultado_ano1']:,.0f}**"
    )
)

| Registro da decisão | |
|---|---|
| Responsável pela aprovação | *preencher* |
| Data | *preencher* |
| Decisão | *copiar da célula acima* |
| Condições ou ressalvas aceitas | *preencher* |

# 5.2 · Revisar o processo

## Análise das fases

| Fase | Notebook | O que entregou | Status |
|---|---|---|---|
| 1 · Entendimento do negócio | Canvas do Problema | contexto, dor, objetivo, critérios de sucesso | parcial: Quadro PBL ainda é rascunho em texto, falta a foto do quadro físico |
| 2 · Entendimento dos dados | 00 · 01 | dicionário, 37% de cancelamento geral, duplicatas, ausentes | ok |
| 3 · Preparação | 02 · 03 | base tratada (119.390 x 37) · MinMax + One-Hot (98 features) | ver lacuna 1 |
| 4 · Modelagem | 04-0 · 04-1 · 04-2 · 04-3 | 8 algoritmos comparados · cotovelo · PCA/UMAP · 523 regras de associação | ver lacunas 2 a 5 |
| 5 · Avaliação | 05 | este notebook | - |

## Lacunas e erros identificados

| # | Lacuna | Onde | Efeito | Correção |
|---|---|---|---|---|
| 1 | 27% das reservas são linhas duplicadas, sem chave única; mantidas por decisão deliberada | 02 | não se sabe se algum segmento é inflado por repetição de carga, em vez de comportamento real | testar a clusterização com e sem duplicatas e comparar o tamanho dos segmentos |
| 2 | parâmetros do K-Means, Mean Shift e DBSCAN foram estimados em amostras de até 20.000 reservas, não na base inteira | 04-0 · 04-1 | o raio e a largura de banda podem não representar bem a cauda da distribuição | reestimar em amostras maiores ou em toda a base, se o tempo permitir |
| 3 | silhueta baixa em todos os algoritmos testados (máximo 0,126, abaixo do corte de 0,25) | 04-0 · 04-1 · 05 | a estrutura geométrica no espaço de 98 colunas é fraca; é por isso que este critério entra como não crítico, não como motivo de reprovação | redução de dimensionalidade (04-2) antes de agrupar, como já é sugerido lá |
| 4 | K-Medoids e hierárquica (Ward) foram excluídos por exigirem memória quadrática em 119.390 reservas | 04-0 | nunca testados nesta base; podem produzir uma partição diferente | rodar em uma amostra de 2.000 a 5.000 reservas, como o Quadro PBL já recomenda |
| 5 | `deposit_type=Non Refund` tem 99,4% de cancelamento, mas fica abaixo do suporte mínimo do 04-3 e nunca vira regra | 04-3 | o padrão mais forte da base quase não aparece documentado nas regras de associação | citado aqui e no Quadro PBL; considerar relatar separadamente, fora da mineração de regras |
| 6 | constantes do projeto (faixa de k, silhueta mínima, teto de ruído) estão repetidas em `04-1` e aqui, sem um `src/config.py` central | 04-1 · 05 | risco de os números divergirem entre notebooks | centralizar as constantes em `src/config.py`, como o template de mercado imobiliário já faz |

## Sumário das descobertas

O que diferencia os segmentos: a importância de cada variável no classificador,
somada por variável de origem (o One-Hot é desfeito), a mesma técnica usada no
`04-2` para a redução de dimensionalidade.

In [ ]:
classificador.fit(X[validos], rotulos)  # treina na carteira inteira para ler as importâncias
variavel = {
    c: max((o for o in df.columns if c == o or c.startswith(f"{o}_")), key=len, default=c)
    for c in base.columns
}  # feature -> variável de origem
importancia = (
    pd.Series(classificador.feature_importances_, index=base.columns)
    .groupby(variavel)
    .sum()
    .sort_values(ascending=False)
)

ax = importancia.head(12)[::-1].plot.barh(figsize=(8, 5), color="tab:blue")
ax.set(title="O que mais diferencia os segmentos", xlabel="importância no classificador")
ax.grid(alpha=0.3, axis="x")
plt.show()

In [ ]:
descobertas = [
    f"A carteira foi dividida em **{k} segmentos** pelo {selecionado['modelo']}; "
    f"o menor tem {fatia_min:.1%} das reservas.",
    f"A taxa de cancelamento vai de **{taxas.min():.1%}** a **{taxas.max():.1%}** "
    f"({razao_cancelamento:.1f}x) entre os segmentos.",
    f"As variáveis que mais separam os segmentos são **{', '.join(importancia.index[:3])}**.",
    f"Uma reserva nova é atribuída ao segmento certo em **{acuracia:.1%}** dos casos.",
    (
        f"A separação geométrica é {'adequada' if silhueta >= SILHUETA_MINIMA else '**fraca**'} "
        f"(silhueta {silhueta:.3f}): os segmentos são faixas de uma carteira contínua, "
        "não grupos isolados."
    ),
]
limitacoes = [
    "A base cobre reservas de 2015 a 2017 de um resort e um hotel urbano específicos: "
    "a representatividade para outra operação precisa ser confirmada.",
    "O modelo descreve a reserva no momento em que foi feita, não o hóspede: duas reservas "
    "do mesmo hóspede podem cair em segmentos diferentes.",
    "O impacto financeiro depende de premissas não validadas (seção 5.1).",
    "Os segmentos refletem as escolhas de pré-processamento do 03 (country agrupado em 16 "
    "categorias, agent/company descartados); com outra escolha, eles podem mudar.",
]
texto_descobertas = "\n".join(f"- {item}" for item in descobertas)
texto_limitacoes = "\n".join(f"- {item}" for item in limitacoes)
display(
    Markdown(
        f"**Descobertas**\n{texto_descobertas}\n\n**Limitações do modelo**\n{texto_limitacoes}"
    )
)

# 5.3 · Determinar as próximas etapas

A próxima ação sai de uma regra declarada antes de olhar o resultado:

| Situação | Próxima ação |
|---|---|
| nenhum critério crítico falha e o cenário base se paga | **Implantação**, com as ressalvas registradas |
| algum critério crítico falha ou o cenário base não se paga, mas a maioria dos critérios é atendida | **Iterar mais**: voltar às fases 3 e 4 com as correções das lacunas |
| a maioria dos critérios falha | **Novo projeto**: revisar a pergunta de negócio ou os dados |

In [ ]:
acoes = {
    "Segmentos operáveis": "restringir k à faixa do negócio na regra de escolha do 04-0",
    "Menor segmento": "subir o tamanho mínimo de segmento ou fundir segmentos pequenos",
    "Cancelamento distingue os segmentos": (
        "agrupar sobre as bases reduzidas do 04-2, que concentram a relevância em menos colunas"
    ),
    "Reserva nova classificável": (
        "treinar e validar um classificador supervisionado sobre os segmentos, como já feito aqui"
    ),
    "Separação dos grupos": "reduzir a dimensionalidade (04-2) e revisar o peso do One-Hot no 03",
    "Cobertura da carteira": (
        "atribuir o ruído ao segmento mais próximo ou trocar o algoritmo de densidade"
    ),
    "Retorno financeiro": (
        "revalidar o ganho de receita e os custos com a operação: no cenário base, "
        "checar se o investimento se paga no ano 1"
    ),
}

if aprovado:
    proxima = "Implantação"
elif avaliacao["atende"].mean() >= 0.5:
    proxima = "Iterar mais"
else:
    proxima = "Novo projeto"

falhas = avaliacao.query("not atende")["critério"].tolist()
if not cenario_base_positivo:
    falhas = falhas + ["Retorno financeiro"]

justificativa = (
    f"{avaliacao['atende'].sum()} de {len(avaliacao)} critérios atendidos; "
    f"críticos com falha: {', '.join(falhas_criticas) or 'nenhum'}; "
    f"resultado do ano 1 no cenário base: ${cenarios.loc['base', 'resultado_ano1']:,.0f}."
)

texto_acoes = "\n".join(f"- *{f}*: {acoes[f]}" for f in falhas)
texto_implantacao = (
    "\n\n**Para a implantação:** expor o classificador para atribuir reservas novas, "
    "publicar as políticas de overbooking por segmento e agendar o re-treino."
)
display(
    Markdown(
        f"### Próxima ação: {proxima}\n**Justificativa:** {justificativa}\n\n"
        + (f"**Ações para cada critério não atendido:**\n{texto_acoes}" if falhas else "")
        + (texto_implantacao if proxima == "Implantação" else "")
    )
)

In [ ]:
(RAIZ / "reports").mkdir(exist_ok=True)
cenarios_dict = cenarios.round(2).reset_index(names="cenario").to_dict(orient="records")
(RAIZ / "reports" / "05-avaliacao.json").write_text(
    json.dumps(
        {
            "modelo": selecionado,
            "criterios": avaliacao.to_dict(orient="records"),
            "cenarios_financeiros": cenarios_dict,
            "aprovado": bool(aprovado),
            "proxima_acao": proxima,
            "justificativa": justificativa,
        },
        ensure_ascii=False,
        indent=2,
        default=lambda v: v.item() if hasattr(v, "item") else str(v),
    ),
    encoding="utf-8",
)
print("Avaliação registrada em reports/05-avaliacao.json")

| Registro da próxima etapa | |
|---|---|
| Decisão | *copiar da célula acima* |
| Justificativa complementar (contexto de negócio) | *preencher* |
| Responsável | *preencher* |
| Prazo da próxima entrega | *preencher* |